In [0]:
# Databricks notebook source
# Gold Layer - fact_sales
# NOTE: run 02_dim_customer and 03_dim_product (and 01_dim_date, whenever the
# data's date range changes) BEFORE this notebook, so every CustomerID/StockCode
# in the current Silver batch already has a surrogate key to join to.

from pyspark.sql import functions as F

# COMMAND ----------

# Structured Streaming source off the Silver Delta table. Delta tables can be
# read as a stream, so this picks up only the new/changed rows since the last
# checkpointed run - consistent with the Bronze/Silver Auto Loader pattern.
silver_stream = spark.readStream.table("retail_sales_dev.silver.sales_cleaned")

# Dimensions are read as static (batch) DataFrames. A streaming-to-static join
# is fully supported by Structured Streaming and does not require foreachBatch,
# because only one side of the join is a streaming source.
dim_customer = spark.table("retail_sales_dev.gold.dim_customer")
# .select(
#     "CustomerID", "CustomerSK"
# )
dim_product = spark.table("retail_sales_dev.gold.dim_product").select(
    "StockCode", "ProductSK"
)
dim_date = spark.table("retail_sales_dev.gold.dim_date").select(
    "CalendarDate", "DateSK"
)

dim_country_currency = spark.table("retail_sales_dev.gold.dim_country_currency")

exchange_rates = spark.table("retail_sales_dev.silver.exchange_rates_cleaned")

In [0]:
invoice_date_only = F.to_date(silver_stream.InvoiceDate)

fact_df = (
    silver_stream
    .join(dim_customer,
    (silver_stream.CustomerID == dim_customer.CustomerID)
    & (invoice_date_only >= dim_customer.StartDate)
    & (
        (invoice_date_only < dim_customer.EndDate)
       | (dim_customer.iscurrent == True)
    ),
    how="left",
    )    
    .join(dim_product, on="StockCode", how="left")
    .join(
        dim_date,
        F.to_date(silver_stream.InvoiceDate) == dim_date.CalendarDate,
        how="left"
    )
    .join(dim_country_currency, on= "Country", how = "left")
    .join(
        exchange_rates,
        (invoice_date_only == exchange_rates.RateDate)
        & (dim_country_currency.CurrencyCode == exchange_rates.TargetCurrency
            ),
        how = "left")
    .select(
        F.col("Invoice").alias("InvoiceNumber"),
        F.col("CustomerSK"),
        F.col("ProductSK"),
        F.col("DateSK"),
        F.col("Quantity"),
        F.col("Price").alias("UnitPrice"),
        (F.col("Quantity") * F.col("Price")).alias("TotalAmount"),
        F.col("CurrencyCode").alias("LocalCurrency"),
        F.col("Rate").alias("ExchangeRate"),
        F.when(
            F.col("CurrencyCode") == "GBP",
            F.col("Quantity") * F.col("Price")
             )
            .otherwise(F.col("Quantity") * F.col("Price") * F.col("Rate"))
            .alias("TotalAmoungLocal"),
        F.col("is_cancelled"),
        F.col("InvoiceDate"),
    )
)


In [0]:

# COMMAND ----------

# fact_df = (
#     silver_stream
#     .join(dim_customer, on="CustomerID", how="left")
#     .join(dim_product, on="StockCode", how="left")
#     .join(
#         dim_date,
#         F.to_date(silver_stream.InvoiceDate) == dim_date.CalendarDate,
#         how="left"
#     )
#     .join(dim_product, on= "StockCode", how= "left")
#     .join(dim_date, invoice_date_only == dim_date.CalendarDate,how = "left")
#     .join(dim_country_currency, on= "Country", how = "left")
#     .join(
#         exchange_rates,
#         (invoice_date_only = exchange_rates.RateDate)
#         & (dim_country.CurrencyCode == exchange_rates.TargetCurrency
#             ),
#         how = "left")
#     .select(
#         F.col("Invoice").alias("InvoiceNumber"),
#         F.col("CustomerSK"),
#         F.col("ProductSK"),
#         F.col("DateSK"),
#         F.col("Quantity"),
#         F.col("Price").alias("UnitPrice"),
#         (F.col("Quantity") * F.col("Price")).alias("TotalAmount"),
#         F.col("CurrencyCode").alias("LocalCurrency"),
#         F.col("Rate").alias("ExchangeRate"),
#         F.when(
#             F.col("CurrencyCode" == "GBP",
#             F.col("Quantity") * F.col("Price")
#              )
#             .otherwise(F.col("Quantity") * F.col("Price") * F.col("Rate"))
#             .alias("TotalAmoungLocal"),
#         F.col("is_cancelled"),
#         F.col("InvoiceDate"),
#     )
# )

# COMMAND ----------

(
    fact_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation","/Volumes/retail_sales_dev/gold/checkpoints/fact_sales/checkpoint")  # match your existing checkpoint root
    .trigger(availableNow=True)
    .toTable("retail_sales_dev.gold.fact_sales")
)

In [0]:
# spark.sql("DROP TABLE IF EXISTS retail_sales_dev.gold.fact_sales1")
# dbutils.fs.rm("/checkpoints/gold_fact_sales/", recurse=True)


In [0]:
silver_table = spark.table("retail_sales_dev.silver.sales_cleaned")
fact_df = (
    silver_table
    .join(dim_customer,
    (silver_table.CustomerID == dim_customer.CustomerID)
    & (silver_table.InvoiceDate >= dim_customer.StartDate)
    # & (
    #     (silver_table.InvoiceDate < dim_customer.EndDate)
    #    | (dim_customer.iscurrent == True)
    # )
    ,
    how="left",
    ))    

# display(fact_df)

In [0]:
# %sql
# UPDATE retail_sales_dev.gold.dim_customer
# SET StartDate = DATE('1900-01-01'),
#     IsCurrent = True
# WHERE StartDate IS NOT NULL;